# Objective

# Setup

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import os
from utils import agop_conv as agc
from utils import trainer as tr
from torch.utils.data import Dataset
import random
import torch.backends.cudnn as cudnn
import numpy as np
from trained_models.MNIST.model2 import trainer as t
import numpy as np
from sklearn.model_selection import train_test_split
from torch.linalg import norm
from torchvision import models
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
from copy import deepcopy

Setting up a new session...
Without the incoming socket you cannot receive events from the server or register event handlers to your Visdom client.


In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
#device='cpu'
print(f"Using device: {device}")

Using device: cuda:0


In [3]:
torch.cuda.empty_cache()

In [4]:
def agop_trajectory(epoch, kwargs):
     corr_list=[]
     for i in range(2):
         G,corr = agc.verify_NFA(kwargs['net'], kwargs['init_net'], kwargs['train_loader'] , layer_idx=i)
         corr_list.append(corr)
     return corr_list
            
    

In [ ]:
pwd = os.getcwd()+'/trajectory.pt'
trainloader, valloader, testloader, init_net, net = t.train_net(force_train=False, fn=agop_trajectory, kwargs={'save_path': pwd})



In [15]:
pwd

'/work/DLR/experiments/MNIST/agop_cnn_trajectory/trajectory.pt'

In [8]:
print(tr.get_acc_ce(net.to(device), testloader))

99.15


In [7]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
matplotlib.use('Agg')
def plot_correlation_trajectory_from_tensors(data):
    """
    Plots the correlation trajectory from data that was originally PyTorch Tensors.
    This function now explicitly handles converting GPU tensors to CPU before 
    extracting the scalar float values, resolving the TypeError.
    
    Args:
        data (dict): Dictionary where keys are epoch numbers (int) and values 
                     are lists of correlation values (torch.Tensor).
    """
    # Sort epochs to ensure chronological order
    epochs = sorted(data.keys())

    # Extract correlations: For each epoch, extract the scalar value (.item()) 
    # from each PyTorch Tensor in the list.
    # FIX: Use .cpu().item() to safely move the tensor to host memory before conversion.
    correlations = [[t.cpu().item() for t in data[e]] for e in epochs]
    
    # Transpose to get correlations grouped by layer (layer-first view)
    correlations_np = np.array(correlations).T
    num_layers = correlations_np.shape[0]

    # Use a distinct, colorblind-friendly color map (e.g., 'Dark2')
    # Use plt.cm.get_cmap which is a reliable way to get an indexed color list
    colors = plt.cm.get_cmap('Dark2', num_layers).colors
    
    plt.figure(figsize=(10, 6))
    
    for i in range(num_layers):
        # Plot correlations for the i-th layer, using a unique color
        plt.plot(epochs, correlations_np[i], 
                 marker='o', 
                 linestyle='-', 
                 markersize=6, 
                 color=colors[i],
                 label=f'Layer {i+1}')

    plt.title('Layer Correlation Trajectory Across Training Epochs', fontsize=16)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Correlation (Pearson)', fontsize=12)
    
    # Ensure all epoch ticks are visible
    plt.xticks(epochs) 
    
    plt.legend(title='Convolutional Layers', loc='best', frameon=True)
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # Adjust y-limit dynamically to zoom in on the data
    min_corr = np.min(correlations_np)
    max_corr = np.max(correlations_np)
    # Set y-limits slightly outside the data range
    plt.ylim(min_corr - 0.005, max_corr + 0.005)
    
    plt.tight_layout()

    # Save the plot
    output_filename = 'layer_correlation_trajectory_non_gui.png'
    plt.savefig(output_filename)
    print(f"Plot saved to {output_filename}")


data= torch.load('trajectory.pt', weights_only= True)
plot_correlation_trajectory_from_tensors(data)


/tmp/ipykernel_2730/2608006201.py:29: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = plt.cm.get_cmap('Dark2', num_layers).colors


Plot saved to layer_correlation_trajectory_non_gui.png
